# 03 - Player Features: Multi-Accounting Group

Purpose: build and verify only the Phase 3 `ma_` feature group from the frozen snapshot.

Input: `frauddet.snapshot.load_snapshot` using `config.yaml` `phase3.input_dir`. No live Mongo and no mutable top-level parquet inputs.

Outputs:
- `data/player_features.parquet`: one row per canonical player, wide scalar + metadata columns.
- `data/player_features_evidence.parquet`: one row per `player_key + feature_name`; `feature_evidence` is JSON reviewer evidence.

Evidence uses direct one-hop links only. Identity hashes remain hashed; no raw NIN/email is present. Tier 4 IP/bank/passport features are deferred.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet.features.multi_accounting import FEATURE_SPECS, build_multi_accounting_features
from frauddet.snapshot import load_snapshot, snapshot_dir

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
print("snapshot_dir", snapshot_dir())

snapshot_dir C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book\data\snapshot_phase3_2026-06-19b


## Build

In [2]:
result = build_multi_accounting_features()
player_features = result.player_features
feature_evidence = result.feature_evidence
result.output_paths, player_features.shape, feature_evidence.shape

({'player_features': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features.parquet'),
  'feature_evidence': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features_evidence.parquet')},
 (116, 37),
 (1044, 3))

## Feature Coverage

In [3]:
coverage_rows = []
for feature_name, spec in FEATURE_SPECS.items():
    values = player_features[feature_name]
    coverage_rows.append(
        {
            "feature_name": feature_name,
            "strength": spec.strength,
            "scoring_role": spec.scoring_role,
            "players": len(values),
            "non_null": int(values.notna().sum()),
            "non_zero": int(values.fillna(0).astype(float).ne(0).sum()),
            "null": int(values.isna().sum()),
        }
    )
coverage = pd.DataFrame(coverage_rows)
display(coverage)

assert int(coverage.loc[coverage.feature_name.eq("ma_email_shared_account_count"), "non_zero"].iloc[0]) == 2
assert int(coverage.loc[coverage.feature_name.eq("ma_nin_shared_account_count"), "non_zero"].iloc[0]) == 0
assert int(coverage.loc[coverage.feature_name.eq("ma_identity_phone_collision_count"), "non_zero"].iloc[0]) == 0
assert player_features["ma_nin_shared_account_count"].notna().all()
assert player_features["ma_identity_phone_collision_count"].notna().all()

,feature_name,strength,scoring_role,players,non_null,non_zero,null
0,ma_nin_shared_account_count,strong,scoring,116,116,0,0
1,ma_email_shared_account_count,strong,scoring,116,116,2,0
2,ma_device_shared_account_count,strong,scoring,116,87,85,29
3,ma_withdrawal_recipient_shared_count,strong,scoring,116,10,0,106
4,ma_identity_phone_collision_count,strong,scoring,116,116,0,0
5,ma_referred_by_linked_account,strong,supporting,116,116,3,0
6,ma_referral_fanout_count,weak,supporting,116,116,2,0
7,ma_cocreated_linked_count,moderate,supporting,116,116,29,0
8,ma_device_count,weak,context_only,116,87,87,29


## Evidence Integrity

In [4]:
evidence_lookup = {
    (row.player_key, row.feature_name): json.loads(row.feature_evidence)
    for row in feature_evidence.itertuples(index=False)
}
for row in player_features.itertuples(index=False):
    values = row._asdict()
    for feature_name, spec in FEATURE_SPECS.items():
        value = values[feature_name]
        if spec.scoring_role == "scoring" and pd.notna(value) and bool(value):
            assert evidence_lookup[(values["player_key"], feature_name)]
print("VERIFIED: every non-zero scoring ma_ feature has non-empty evidence")

VERIFIED: every non-zero scoring ma_ feature has non-empty evidence


## Hand Reconciliation Against Frozen Parquet

In [5]:
players_raw = load_snapshot("players")
logins_raw = load_snapshot("logins")
money_raw = load_snapshot("money")

email_players = player_features.loc[
    player_features["ma_email_shared_account_count"].gt(0), "player_key"
].astype(str).tolist()
referral_players = player_features.loc[
    player_features["ma_referred_by_linked_account"].fillna(False), "player_key"
].astype(str).tolist()
picked_players = list(dict.fromkeys(email_players + referral_players[:1]))[:3]

print("picked_players", picked_players)
display(
    player_features.loc[
        player_features.player_key.isin(picked_players),
        ["player_key"] + list(FEATURE_SPECS),
    ]
)

focus_features = {
    "ma_email_shared_account_count",
    "ma_referred_by_linked_account",
}
focus_evidence = feature_evidence[
    feature_evidence.player_key.isin(picked_players)
    & feature_evidence.feature_name.isin(focus_features)
].copy()
focus_evidence["parsed_evidence"] = focus_evidence.feature_evidence.map(json.loads)
display(focus_evidence[["player_key", "feature_name", "parsed_evidence"]])

linked_players = set(picked_players)
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        linked_players.update(item.get("other_player_keys", []))
        if item.get("referrer_player_key"):
            linked_players.add(item["referrer_player_key"])

# Raw frozen rows needed to reconcile the selected hashes/referrals/devices.
display(
    players_raw.loc[
        players_raw.player_key.isin(linked_players),
        ["player_key", "created_at", "nin_hash", "email_hash", "referred_by_key"],
    ].sort_values("player_key")
)

shared_fingerprints = set()
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        if item.get("shared_key_type") == "fingerprint":
            shared_fingerprints.add(item["shared_key"])
        for linkage in item.get("corroborating_linkages", []):
            if linkage.get("shared_key_type") == "fingerprint":
                shared_fingerprints.add(linkage["shared_key"])

display(
    logins_raw.loc[
        logins_raw.player_key.isin(linked_players)
        & logins_raw.fingerprint.isin(shared_fingerprints),
        ["player_key", "source_id", "fingerprint", "created_at"],
    ].drop_duplicates().sort_values(["fingerprint", "player_key", "created_at"])
)

# No shared-recipient feature fires in this snapshot, but show picked players' completed withdrawals if any.
display(
    money_raw.loc[
        money_raw.player_key.isin(picked_players)
        & money_raw.txn_type.eq("WITHDRAWAL")
        & money_raw.is_money_out.fillna(False),
        ["player_key", "transaction_id", "recipient_normalized", "requested_at"],
    ].sort_values(["player_key", "requested_at"])
)

picked_players ['6a26959dfc099dd782863ecf', '6a311326aa7ed994dfda83af', '6a0f0df7a5d7ce78c657a62f']


,player_key,ma_nin_shared_account_count,ma_email_shared_account_count,ma_device_shared_account_count,ma_withdrawal_recipient_shared_count,ma_identity_phone_collision_count,ma_referred_by_linked_account,ma_referral_fanout_count,ma_cocreated_linked_count,ma_device_count
8,6a0f0df7a5d7ce78c657a62f,0,0,27,<NA>,0,True,0,0,2
41,6a26959dfc099dd782863ecf,0,1,34,<NA>,0,False,0,0,2
84,6a311326aa7ed994dfda83af,0,1,10,0,0,False,0,0,2


,player_key,feature_name,parsed_evidence
75,6a0f0df7a5d7ce78c657a62f,ma_email_shared_account_count,[]
79,6a0f0df7a5d7ce78c657a62f,ma_referred_by_linked_account,"[{'corroborating_linkages': [{'shared_key': '48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55', 'shared_key_type': 'fingerprint'}], 'referre..."
372,6a26959dfc099dd782863ecf,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a311326aa7ed994dfda83af': []}, 'other_player_keys': ['6a311326aa7ed994dfda83af'], 'shared_key': '8e922392304ca6feb6fc074475..."
376,6a26959dfc099dd782863ecf,ma_referred_by_linked_account,[]
759,6a311326aa7ed994dfda83af,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a26959dfc099dd782863ecf': []}, 'other_player_keys': ['6a26959dfc099dd782863ecf'], 'shared_key': '8e922392304ca6feb6fc074475..."
763,6a311326aa7ed994dfda83af,ma_referred_by_linked_account,[]


,player_key,created_at,nin_hash,email_hash,referred_by_key
7,6a0ef9cca5d7ce78c65765d9,2026-05-21 12:25:48.725000+00:00,NaN,ee5fa051aabbad3d0458007a4be7246232cdff2cb260811e0038381518c7ffdb,NaN
8,6a0f0df7a5d7ce78c657a62f,2026-05-21 13:51:51.182000+00:00,NaN,3f6d4dccfe89d1ac131b4a3f3d34cb5fd693ba89050e73111567c1993927d598,6a0ef9cca5d7ce78c65765d9
41,6a26959dfc099dd782863ecf,2026-06-08 10:12:45.844000+00:00,NaN,8e922392304ca6feb6fc07447532165585fc4514d479a73c078b5de238db8e92,NaN
84,6a311326aa7ed994dfda83af,2026-06-16 09:11:02.633000+00:00,4e8604fa51509d5e9fb79a9db6200cb938f934dfe1631b51f72dff668ad3ee9e,8e922392304ca6feb6fc07447532165585fc4514d479a73c078b5de238db8e92,NaN


,player_key,source_id,fingerprint,created_at
54,6a0ef9cca5d7ce78c65765d9,6a0ef9d1a5d7ce78c65765f2,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55,2026-05-21 12:25:53.905000+00:00
108,6a0ef9cca5d7ce78c65765d9,6a102f74a5d7ce78c657fec4,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55,2026-05-22 10:27:00.477000+00:00
107,6a0f0df7a5d7ce78c657a62f,6a102f1da5d7ce78c657fc66,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55,2026-05-22 10:25:33.425000+00:00
617,6a26959dfc099dd782863ecf,6a2695aefc099dd782863ef0,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55,2026-06-08 10:13:02.721000+00:00
633,6a26959dfc099dd782863ecf,6a26a71efc099dd782865436,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55,2026-06-08 11:27:26.011000+00:00


,player_key,transaction_id,recipient_normalized,requested_at
1138,6a311326aa7ed994dfda83af,TXNY_MQGIA50Q_275478271E03,755432100,2026-06-16 10:35:15.894000+00:00


## FINDINGS — Multi-accounting features — frozen Phase 3 snapshot — 2026-06-22
- Verified: `player_features.parquet` has one row for each of 116 canonical snapshot players and contains only the nine requested Tier 1-3 `ma_` features plus per-feature null/strength/role metadata.
- Verified: evidence store schema is long-form `(player_key, feature_name, feature_evidence)` with JSON evidence. It contains 1,044 rows (116 players x 9 features); non-zero scoring features always have evidence.
- Verified: `ma_nin_shared_account_count` is non-null for 116/116 and non-zero for 0; absent NIN correctly maps to measured `0`, not null.
- Verified: `ma_email_shared_account_count` is non-null for 116/116 and non-zero for 2, representing the single frozen shared-email pair.
- Verified: `ma_device_shared_account_count` is measurable for 87/116 and non-zero for 85; 29 players are null with `no_valid_fingerprint_logins`. The high sharing is the known office-device artifact.
- Verified: `ma_withdrawal_recipient_shared_count` is measurable for 10/116 and non-zero for 0; 106 players are null with `no_completed_withdrawals`.
- Verified: `ma_identity_phone_collision_count` is non-null for 116/116 and non-zero for 0, confirming the dormant phone-collision path is wired.
- Verified: `ma_referred_by_linked_account` is non-null for 116/116 and non-zero for 3; current hits are corroborated by shared device evidence and should be treated as dev office artifacts.
- Verified: `ma_referral_fanout_count` is non-null for 116/116 and non-zero for 2.
- Verified: `ma_cocreated_linked_count` is non-null for 116/116 and non-zero for 29 using the configured 15-minute placeholder window plus a direct shared signal.
- Verified: `ma_device_count` is measurable/non-zero for 87/116; 29 are null with `no_valid_fingerprint_logins`.
- Surprises: `ma_referred_by_linked_account` is not dormant in the frozen snapshot because referral pairs also share the dev office fingerprint; mechanism works, calibration remains production-only.
- Gaps: `players.parquet` contains no raw identity values by design; reviewer evidence can show hashes but cannot reconstruct raw NIN/email. Tier 4 IP/bank/passport features remain documented TODOs only.
- Decisions needed: None for the `ma_` group contract. Review before proceeding to `pay_`.
- Updated assumptions: All Phase 3 feature computation and hand reconciliation read only through the frozen snapshot loader; no live Mongo or mutable top-level input parquet is used.